In [ ]:
!nvidia-smi
!python --version
!pip show transformers torch gradio datasets 2>/dev/null | grep -E "^(Name|Version)"

In [ ]:
import time

import torch
from transformers import BlipForConditionalGeneration, BlipProcessor

MODEL_ID = "Salesforce/blip-image-captioning-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

start = time.perf_counter()
processor = BlipProcessor.from_pretrained(MODEL_ID)
model = BlipForConditionalGeneration.from_pretrained(MODEL_ID)
model.to(device)
model.eval()
print("model load seconds:", round(time.perf_counter() - start, 1))

In [ ]:
import gradio as gr
from PIL import Image


def caption_image(image: Image.Image | None) -> str:
    if image is None:
        return "Please upload a photo first."
    rgb_image = image.convert("RGB")
    inputs = processor(images=rgb_image, return_tensors="pt")
    inputs = inputs.to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=40)
    return processor.decode(output_ids[0], skip_special_tokens=True)


demo = gr.Interface(
    fn=caption_image,
    inputs=gr.Image(type="pil", label="Your photo", sources=["upload"]),
    outputs=gr.Textbox(label="Caption", lines=2),
    title="Sightcast",
    description="Upload a photo and get a short description.",
)

demo.launch()

In [ ]:
import io
import time
import zipfile

import numpy as np
import torch
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, SpeechT5Processor

SPEECH_MODEL_ID = "microsoft/speecht5_tts"
VOCODER_ID = "microsoft/speecht5_hifigan"
VOICE_PREFIX = "spkrec-xvect/cmu_us_slt_arctic"
TEST_TEXT = "The chat screen with the chat button highlighted."

start = time.perf_counter()
speech_processor = SpeechT5Processor.from_pretrained(SPEECH_MODEL_ID)
speech_model = SpeechT5ForTextToSpeech.from_pretrained(SPEECH_MODEL_ID)
vocoder = SpeechT5HifiGan.from_pretrained(VOCODER_ID)
speech_model.to(device)
vocoder.to(device)
speech_model.eval()
vocoder.eval()
print("speech load seconds:", round(time.perf_counter() - start, 1))

zip_path = hf_hub_download(
    repo_id="Matthijs/cmu-arctic-xvectors",
    repo_type="dataset",
    filename="spkrec-xvect.zip",
)
with zipfile.ZipFile(zip_path) as archive:
    voice_names = []
    for name in archive.namelist():
        if name.startswith(VOICE_PREFIX):
            voice_names.append(name)
    voice_names.sort()
    voice_bytes = archive.read(voice_names[0])
print("voice file:", voice_names[0])

voice_array = np.load(io.BytesIO(voice_bytes))
print("voice shape:", voice_array.shape)
speaker_embeddings = torch.tensor(voice_array, dtype=torch.float32)
speaker_embeddings = speaker_embeddings.reshape(1, -1).to(device)

inputs = speech_processor(text=TEST_TEXT, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)

start = time.perf_counter()
with torch.no_grad():
    speech = speech_model.generate_speech(
        input_ids,
        speaker_embeddings,
        vocoder=vocoder,
    )
speech_seconds = time.perf_counter() - start

waveform = speech.cpu().numpy()
sampling_rate = vocoder.config.sampling_rate
print("speech seconds:", round(speech_seconds, 2))
print("audio length seconds:", round(len(waveform) / sampling_rate, 2))
display(Audio(waveform, rate=sampling_rate))

In [ ]:
import numpy as np


def tidy_caption(text: str) -> str:
    cleaned = text.strip()
    if cleaned == "":
        return cleaned
    cleaned = cleaned[0].upper() + cleaned[1:]
    if cleaned[-1] not in ".!?":
        cleaned = cleaned + "."
    return cleaned


def speak(text: str) -> tuple[int, np.ndarray]:
    inputs = speech_processor(text=text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)
    with torch.no_grad():
        speech = speech_model.generate_speech(
            input_ids,
            speaker_embeddings,
            vocoder=vocoder,
        )
    waveform = speech.cpu().numpy()
    clipped = np.clip(waveform, -1.0, 1.0)
    pcm = (clipped * 32767).astype(np.int16)
    return vocoder.config.sampling_rate, pcm


def narrate(image: Image.Image | None) -> tuple[str, tuple[int, np.ndarray] | None]:
    if image is None:
        return "Please upload a photo first.", None
    caption = tidy_caption(caption_image(image))
    if caption == "":
        return "No caption was produced.", None
    audio = speak(caption)
    return caption, audio


demo = gr.Interface(
    fn=narrate,
    inputs=gr.Image(type="pil", label="Your photo", sources=["upload"]),
    outputs=[
        gr.Textbox(label="Caption", lines=2),
        gr.Audio(type="numpy", label="Spoken description"),
    ],
    title="Sightcast",
    description="Upload a photo. Get a caption and hear it read aloud.",
    flagging_mode="never",
)

demo.launch()